In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
SOURCE_VIDEO_PATH = DATA_DIR / "Bad_detections_game2.mp4"
DURATION_S = 30

if not SOURCE_VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {SOURCE_VIDEO_PATH}\n"
        "Put a clip in data/ or change SOURCE_VIDEO_PATH."
    )
print(SOURCE_VIDEO_PATH)

In [2]:
import os

os.environ["CORE_MODEL_SAM_ENABLED"] = "False"
os.environ["CORE_MODEL_SAM3_ENABLED"] = "False"
os.environ["CORE_MODEL_YOLO_WORLD_ENABLED"] = "False"
os.environ["CORE_MODEL_GAZE_ENABLED"] = "False"
os.environ.setdefault(
    "ONNXRUNTIME_EXECUTION_PROVIDERS",
    "CUDAExecutionProvider,CPUExecutionProvider",
)

from src.utils.env import load_api_keys

load_api_keys()

In [ ]:
import numpy as np
import pandas as pd
import supervision as sv
from inference import get_model
from tqdm import tqdm
from sports import (
    clean_paths,
    ConsecutiveValueTracker,
    TeamClassifier,
    MeasurementUnit,
    ViewTransformer,
)
from sports.basketball import (
    CourtConfiguration,
    League,
    draw_court,
    draw_points_on_court,
    draw_paths_on_court,
    draw_made_and_miss_on_court,
)


In [ ]:
PLAYER_DETECTION_MODEL_ID = "basketball-player-detection-3-ycjdo/13"
PLAYER_DETECTION_MODEL_CONFIDENCE = 0.4
PLAYER_DETECTION_MODEL_IOU_THRESHOLD = 0.9
PLAYER_DETECTION_MODEL = get_model(
    model_id=PLAYER_DETECTION_MODEL_ID,
    api_key=os.environ["API_KEY"],
)

KEYPOINT_DETECTION_MODEL_ID = "basketball-court-detection-2/14"
KEYPOINT_DETECTION_MODEL_CONFIDENCE = 0.3
KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE = 0.5
KEYPOINT_DETECTION_MODEL = get_model(
    model_id=KEYPOINT_DETECTION_MODEL_ID,
    api_key=os.environ["API_KEY"],
)
KEYPOINT_COLOR = sv.Color.from_hex('#FF1493')

COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

### Check frame

In [ ]:
PLAYER_JUMP_SHOT_CLASS_ID = 5
FRAME_IDX = 730
i=0

box_annotator = sv.BoxAnnotator(color=COLOR, thickness=2)
label_annotator = sv.LabelAnnotator(color=COLOR, text_color=sv.Color.BLACK)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=FRAME_IDX-5, iterative_seek=True)

for frame in frame_generator:
    frame = next(frame_generator)
    i += 1
    if i > 10:
        break

    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE, iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result)
    detections = detections[detections.class_id == PLAYER_JUMP_SHOT_CLASS_ID]

    annotated_frame = frame.copy()
    annotated_frame = box_annotator.annotate(
        scene=annotated_frame,
        detections=detections)
    annotated_frame = label_annotator.annotate(
        scene=annotated_frame,
        detections=detections)

    sv.plot_image(annotated_frame)

### Cursor Shot Mapper

In [38]:
TEAM_ROSTERS = {
  "New York Knicks": {
    "55": "Hukporti",
    "1": "Payne",
    "0": "Wright",
    "11": "Brunson",
    "3": "Hart",
    "32": "Towns",
    "44": "Shamet",
    "25": "Bridges",
    "2": "McBride",
    "23": "Robinson",
    "8": "Anunoby",
    "4": "Dadiet",
    "5": "Achiuwa",
    "13": "Kolek"
  },
  "Boston Celtics": {
    "42": "Horford",
    "55": "Scheierman",
    "9": "White",
    "20": "Davison",
    "7": "Brown",
    "0": "Tatum",
    "27": "Walsh",
    "4": "Holiday",
    "8": "Porzingis",
    "40": "Kornet",
    "88": "Queta",
    "11": "Pritchard",
    "30": "Hauser",
    "12": "Craig",
    "26": "Tillman"
  }
}

TEAM_COLORS = {
    "New York Knicks": "#006BB6",
    "Boston Celtics": "#007A33"
}

In [ ]:
STRIDE = 30
PLAYER_CLASS_IDS = [3, 4, 5, 6, 7] # player, player-in-possession, player-jump-shot, player-layup-dunk, player-shot-block
crops = []
SOURCE_VIDEO_DIRECTORY = "C:/Users/zecab/.gemini/antigravity-ide/scratch/Basketball-player-tracking/data"

for video_path in sv.list_files_with_extensions(SOURCE_VIDEO_DIRECTORY, extensions=["mp4", "avi", "mov"]):
    frame_generator = sv.get_video_frames_generator(source_path=video_path, stride=STRIDE)

    for frame in tqdm(frame_generator):

        result = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE, iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD, class_agnostic_nms=True)[0]
        detections = sv.Detections.from_inference(result)
        detections = detections[np.isin(detections.class_id, PLAYER_CLASS_IDS)]

        boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=0.4)
        for box in boxes:
            crops.append(sv.crop_image(frame, box))

In [ ]:
# define team annotators
mask_annotator = sv.MaskAnnotator(
    color=COLOR,
    color_lookup=sv.ColorLookup.TRACK,
    opacity=0.5)
box_annotator = sv.BoxAnnotator(
    color=COLOR,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2
)
id_annotator = sv.LabelAnnotator(
    color=COLOR,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8
)

# we use RF-DETR model to aquire future SAM2 prompt

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE, iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
detections = sv.Detections.from_inference(result)
detections = detections[np.isin(detections.class_id, PLAYER_CLASS_IDS)]
detections.tracker_id = np.arange(1, len(detections.class_id) + 1)

annotated_frame = frame.copy()
annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
annotated_frame = id_annotator.annotate(scene=annotated_frame, detections=detections, labels=detections.tracker_id)


sv.plot_image(annotated_frame)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TeamClassifier device: {device}")
team_classifier = TeamClassifier(device=device)
team_classifier.fit(crops)

In [ ]:
teams = team_classifier.predict(crops)

team_0 = [crop for crop, team in zip(crops, teams) if team == 0]
team_1 = [crop for crop, team in zip(crops, teams) if team == 1]

sv.plot_images_grid(
    images=team_0[:50],
    grid_size=(5, 10),
    size=(10, 5)
)

sv.plot_images_grid(
    images=team_1[:50],
    grid_size=(5, 10),
    size=(10, 5)
)

In [ ]:
# TEAM_NAMES = {
#     0: "New York Knicks",
#     1: "Boston Celtics",
# }
TEAM_NAMES = {
    0: "Boston Celtics",
    1: "New York Knicks",
}

In [ ]:
team_validator = ConsecutiveValueTracker(n_consecutive=1)
# we determine the team for each player and assign a team ID to every detection

boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=0.4)
crops = [sv.crop_image(frame, box) for box in boxes]
TEAMS = np.array(team_classifier.predict(crops))

team_validator.update(tracker_ids=detections.tracker_id, values=TEAMS)

In [ ]:
config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=FRAME_IDX, iterative_seek=True)
frame = next(frame_generator)

# we use a RF-DETR model to detect players

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE, iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
detections = sv.Detections.from_inference(result)
detections = detections[detections.class_id == PLAYER_JUMP_SHOT_CLASS_ID]

team_validator = ConsecutiveValueTracker(n_consecutive=1)
# we determine the team for each player and assign a team ID to every detection

boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=0.4)
crops = [sv.crop_image(frame, box) for box in boxes]
TEAMS = np.array(team_classifier.predict(crops))

team_validator.update(tracker_ids=detections.tracker_id, values=TEAMS)

# we use a keypoint model to detect court landmarks.
# Infer returns only detected landmarks (e.g. 13), while the court has 33
# vertices — match them by class_id, not a boolean mask.

result = KEYPOINT_DETECTION_MODEL.infer(frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
key_points = sv.KeyPoints.from_inference(result)
payload = result.dict(exclude_none=True, by_alias=True) if hasattr(result, "dict") else result
raw_keypoints = (payload.get("predictions") or [{}])[0].get("keypoints") or []
vertices = np.asarray(config.vertices, dtype=float)

frame_landmarks, vertex_ids = [], []
for i, kp in enumerate(raw_keypoints):
    if float(kp.get("confidence", 0.0)) <= KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE:
        continue
    class_id = kp.get("class_id")
    if class_id is None:
        class_id = i if len(raw_keypoints) == len(vertices) else None
    else:
        class_id = int(class_id)
    if class_id is None or class_id < 0 or class_id >= len(vertices):
        continue
    vertex_ids.append(class_id)
    frame_landmarks.append([float(kp["x"]), float(kp["y"])])

if len(vertex_ids) >= 4:

    # we calculate homography matrix

    frame_landmarks = np.asarray(frame_landmarks, dtype=float)
    court_landmarks = vertices[np.asarray(vertex_ids, dtype=int)]

    frame_to_court_transformer = ViewTransformer(
        source=frame_landmarks,
        target=court_landmarks,
    )

    frame_xy = detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)

    # transform video frame coordinates into court coordinates

    court_xy = frame_to_court_transformer.transform_points(points=frame_xy)

    court = draw_made_and_miss_on_court(
        config=config,
        miss_xy=court_xy,
        miss_color = sv.Color.from_hex("#850101"),
        miss_size= 10,
        made_size=25,
        made_color=sv.Color.from_hex("#007A33"),
        made_thickness=6,
        miss_thickness=6,
        line_thickness=4
    )

    sv.plot_image(court)


### Offense-aware ShotEventTracker

Local copy of the sports tracker. Layup/dunk starts only if `layup_is_offense` is true (player jersey matches the team last seen with the ball).


In [ ]:
from enum import Enum
from dataclasses import dataclass
from typing import Optional, List, Literal, TypedDict

import numpy as np
import supervision as sv


BALL_IN_BASKET_MIN_CONSECUTIVE_FRAMES = 2
JUMP_SHOT_MIN_CONSECUTIVE_FRAMES = 3
LAYUP_DUNK_MIN_CONSECUTIVE_FRAMES = 3


class ShotType(Enum):
    NONE = "NONE"
    JUMP = "JUMP"
    LAYUP = "LAYUP"


class ShotEvent(Enum):
    START = "START"
    MADE = "MADE"
    MISSED = "MISSED"


class ShotEventRecord(TypedDict):
    event: Literal["START", "MADE", "MISSED"]
    frame: int
    type: Literal["NONE", "JUMP", "LAYUP"]


def majority_team_id(team_ids) -> Optional[int]:
    values = np.asarray(team_ids)
    values = values[values >= 0]
    if len(values) == 0:
        return None
    unique, counts = np.unique(values, return_counts=True)
    return int(unique[np.argmax(counts)])


def predict_detection_teams(frame, detections, team_classifier, crop_scale=0.4) -> np.ndarray:
    teams = np.full(len(detections), -1, dtype=int)
    if len(detections) == 0:
        return teams
    boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=crop_scale)
    crops, indices = [], []
    for i, box in enumerate(boxes):
        crop = sv.crop_image(frame, box)
        if crop is None or getattr(crop, "size", 0) == 0:
            continue
        crops.append(crop)
        indices.append(i)
    if not crops:
        return teams
    predicted = np.asarray(team_classifier.predict(crops), dtype=int)
    teams[np.asarray(indices, dtype=int)] = predicted
    return teams


def filter_detections_by_team(detections, team_ids, team_id) -> sv.Detections:
    if len(detections) == 0 or team_id is None:
        return detections[np.zeros(len(detections), dtype=bool)]
    return detections[np.asarray(team_ids) == int(team_id)]


def update_offensive_team_id(current, possession_team_ids) -> Optional[int]:
    inferred = majority_team_id(possession_team_ids)
    return inferred if inferred is not None else current


@dataclass
class ShotEventTracker:
    reset_time_frames: int
    minimum_frames_between_starts: int
    cooldown_frames_after_made: int

    shot_in_progress: bool = False
    shot_type: ShotType = ShotType.NONE
    shot_start_frame: Optional[int] = None
    shot_deadline_frame: Optional[int] = None
    frames_since_start: int = 0

    consecutive_jump_shot_frames: int = 0
    consecutive_layup_frames: int = 0
    consecutive_ball_in_basket_frames: int = 0

    last_made_frame: Optional[int] = None

    def update(
        self,
        frame_index: int,
        has_jump_shot: bool,
        has_layup_dunk: bool,
        has_ball_in_basket: bool,
        layup_is_offense: bool = True,
    ) -> List[ShotEventRecord]:
        events: List[ShotEventRecord] = []

        # Layup/dunk only counts if the player is on the team with the ball.
        if not layup_is_offense:
            has_layup_dunk = False

        self.consecutive_jump_shot_frames = self._updated_consecutive_frames(
            self.consecutive_jump_shot_frames, has_jump_shot
        )
        self.consecutive_layup_frames = self._updated_consecutive_frames(
            self.consecutive_layup_frames, has_layup_dunk
        )
        self.consecutive_ball_in_basket_frames = self._updated_consecutive_frames(
            self.consecutive_ball_in_basket_frames, has_ball_in_basket
        )

        reached_jump_shot_threshold = (
            self.consecutive_jump_shot_frames == JUMP_SHOT_MIN_CONSECUTIVE_FRAMES
        )
        reached_layup_threshold = (
            self.consecutive_layup_frames == LAYUP_DUNK_MIN_CONSECUTIVE_FRAMES
        )
        should_start_shot = reached_jump_shot_threshold or reached_layup_threshold

        if should_start_shot and self._within_post_made_cooldown(frame_index):
            should_start_shot = False

        if should_start_shot:
            if self.shot_in_progress:
                if self.frames_since_start >= self.minimum_frames_between_starts:
                    events.append(self._missed_event(frame_index))
                    self._reset_shot_state()
                else:
                    should_start_shot = False

            if should_start_shot:
                shot_type = ShotType.JUMP if reached_jump_shot_threshold else ShotType.LAYUP
                self._start_new_shot(shot_type, frame_index)
                events.append(self._start_event(frame_index))

        if self.shot_in_progress:
            self.frames_since_start += 1

            if self._has_confirmed_make():
                events.append(self._made_event(frame_index))
                self.last_made_frame = frame_index
                self._reset_shot_state()
                return events

            if self._deadline_reached(frame_index):
                events.append(self._missed_event(frame_index))
                self._reset_shot_state()
                return events

        return events

    @staticmethod
    def _updated_consecutive_frames(current_count: int, detected: bool) -> int:
        return current_count + 1 if detected else 0

    def _start_new_shot(self, shot_type: ShotType, frame_index: int) -> None:
        self.shot_in_progress = True
        self.shot_type = shot_type
        self.shot_start_frame = frame_index
        self.shot_deadline_frame = frame_index + self.reset_time_frames
        self.frames_since_start = 0
        self.consecutive_ball_in_basket_frames = 0

    def _has_confirmed_make(self) -> bool:
        return (
            self.consecutive_ball_in_basket_frames
            >= BALL_IN_BASKET_MIN_CONSECUTIVE_FRAMES
        )

    def _deadline_reached(self, frame_index: int) -> bool:
        return (
            self.shot_deadline_frame is not None
            and frame_index >= self.shot_deadline_frame
        )

    def _within_post_made_cooldown(self, frame_index: int) -> bool:
        if self.last_made_frame is None:
            return False
        return (frame_index - self.last_made_frame) < self.cooldown_frames_after_made

    def _start_event(self, frame_index: int) -> ShotEventRecord:
        return {"event": ShotEvent.START.value, "frame": frame_index, "type": self.shot_type.value}

    def _made_event(self, frame_index: int) -> ShotEventRecord:
        return {"event": ShotEvent.MADE.value, "frame": frame_index, "type": self.shot_type.value}

    def _missed_event(self, frame_index: int) -> ShotEventRecord:
        return {"event": ShotEvent.MISSED.value, "frame": frame_index, "type": self.shot_type.value}

    def _reset_consecutive_counters(self) -> None:
        self.consecutive_jump_shot_frames = 0
        self.consecutive_layup_frames = 0
        self.consecutive_ball_in_basket_frames = 0

    def _reset_shot_state(self) -> None:
        self.shot_in_progress = False
        self.shot_type = ShotType.NONE
        self.shot_start_frame = None
        self.shot_deadline_frame = None
        self.frames_since_start = 0
        self._reset_consecutive_counters()


In [ ]:
BALL_IN_BASKET_CLASS_ID = 1
JUMP_SHOT_CLASS_ID = 5
LAYUP_DUNK_CLASS_ID = 6
PLAYER_IN_POSSESSION_CLASS_ID = 4
PLAYER_CLASS_ID = [0,1, 3, 4, 5, 6, 7, 9] # ball, ball_in_rim, player, player-in-possession, player-jump-shot, player-layup-dunk, player-shot-block, ball

team_validator = ConsecutiveValueTracker(n_consecutive=1)
# we determine the team for each player and assign a team ID to every detection

boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=0.4)
crops = [sv.crop_image(frame, box) for box in boxes]
TEAMS = np.array(team_classifier.predict(crops))

config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)
vertex_annotator = sv.VertexAnnotator(color=KEYPOINT_COLOR, radius=8)
box_annotator = sv.BoxAnnotator(color=COLOR, thickness=1)
label_annotator = sv.LabelAnnotator(color=COLOR, text_color=sv.Color.BLACK)

# Team annotators
team_colors = sv.ColorPalette.from_hex([
    TEAM_COLORS[TEAM_NAMES[0]],
    TEAM_COLORS[TEAM_NAMES[1]],
])

team_mask_annotator = sv.MaskAnnotator(
    color=team_colors,
    opacity=0.5,
    color_lookup=sv.ColorLookup.INDEX,
)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=3)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)

shot_event_tracker = ShotEventTracker(
    reset_time_frames=int(video_info.fps * 1.7),
    minimum_frames_between_starts=int(video_info.fps * 0.5),
    cooldown_frames_after_made=int(video_info.fps * 0.5),
)

# Last team seen with the ball. Possession is sparse, so keep it across frames.
offensive_team_id = None


def map_shooter_to_court(frame, detections):
    """Detect court landmarks and map shooter feet into court coordinates."""
    result = KEYPOINT_DETECTION_MODEL.infer(
        frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE
    )[0]
    key_points = sv.KeyPoints.from_inference(result)
    payload = result.dict(exclude_none=True, by_alias=True) if hasattr(result, "dict") else result
    raw_keypoints = (payload.get("predictions") or [{}])[0].get("keypoints") or []
    vertices = np.asarray(config.vertices, dtype=float)

    frame_landmarks, vertex_ids = [], []
    for i, kp in enumerate(raw_keypoints):
        if float(kp.get("confidence", 0.0)) <= KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE:
            continue
        class_id = kp.get("class_id")
        if class_id is None:
            class_id = i if len(raw_keypoints) == len(vertices) else None
        else:
            class_id = int(class_id)
        if class_id is None or class_id < 0 or class_id >= len(vertices):
            continue
        vertex_ids.append(class_id)
        frame_landmarks.append([float(kp["x"]), float(kp["y"])])

    if len(vertex_ids) < 4 or len(detections) == 0:
        return key_points, None

    frame_to_court_transformer = ViewTransformer(
        source=np.asarray(frame_landmarks, dtype=float),
        target=vertices[np.asarray(vertex_ids, dtype=int)],
    )
    frame_xy = detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
    court_xy = frame_to_court_transformer.transform_points(points=frame_xy)
    return key_points, court_xy


pending_shot = None
shots = []

for frame_index, frame in enumerate(tqdm(frame_generator, desc="shot events")):

    # we use a RF-DETR model to detect jump shot, layup, dunk and ball in basket

    result = PLAYER_DETECTION_MODEL.infer(
        frame,
        confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
        iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD,
    )[0]
    detections = sv.Detections.from_inference(result)
    detections = detections[np.isin(detections.class_id, PLAYER_CLASS_ID)]

    possession_dets = detections[detections.class_id == PLAYER_IN_POSSESSION_CLASS_ID]
    if len(possession_dets) > 0:
        offensive_team_id = update_offensive_team_id(
            offensive_team_id,
            predict_detection_teams(frame, possession_dets, team_classifier),
        )

    jump_dets = detections[detections.class_id == JUMP_SHOT_CLASS_ID]
    layup_dets = detections[detections.class_id == LAYUP_DUNK_CLASS_ID]
    if len(layup_dets) > 0:
        layup_teams = predict_detection_teams(frame, layup_dets, team_classifier)
        layup_dets = filter_detections_by_team(
            layup_dets, layup_teams, offensive_team_id
        )

    has_jump_shot = len(jump_dets) > 0
    has_layup_dunk = len(layup_dets) > 0
    has_ball_in_basket = len(detections[detections.class_id == BALL_IN_BASKET_CLASS_ID]) > 0

    events = shot_event_tracker.update(
        frame_index=frame_index,
        has_jump_shot=has_jump_shot,
        has_layup_dunk=has_layup_dunk,
        has_ball_in_basket=has_ball_in_basket,
        layup_is_offense=has_layup_dunk,
    )

    if not events:
        continue

    print(events)

    for event in events:
        kind = event["event"]

        if kind == "START":
            event_dets = jump_dets if event["type"] == "JUMP" else layup_dets
            key_points, court_xy = map_shooter_to_court(frame, event_dets)

            court_x, court_y = np.nan, np.nan
            if court_xy is not None and len(court_xy) > 0:
                court_x, court_y = float(court_xy[0][0]), float(court_xy[0][1])

            if court_x < 0 or court_y < 0:
                print(f"BUGGED MAPPING for {event['type']} frame={event['frame']}")
                continue

            pending_shot = {
                "start_frame": int(event["frame"]),
                "end_frame": None,
                "outcome": None,
                "shot_type": event["type"].lower(),
                "court_x": court_x,
                "court_y": court_y,
            }

            annotated_frame = frame.copy()
            if len(key_points) > 0:
                annotated_frame = vertex_annotator.annotate(
                    scene=annotated_frame,
                    key_points=key_points,
                )
            annotated_frame = box_annotator.annotate(
                scene=annotated_frame,
                detections=event_dets,
            )
            annotated_frame = label_annotator.annotate(
                scene=annotated_frame,
                detections=event_dets,
            )
            if np.isfinite(court_x) and np.isfinite(court_y):
                print(
                    f"START {event['type']} frame={event['frame']} "
                    f"court=({court_x:.1f}, {court_y:.1f})"
                )
            else:
                print(f"START {event['type']} frame={event['frame']} court=unmapped")
            sv.plot_image(annotated_frame)

        elif kind in {"MADE", "MISSED"}:
            row = pending_shot if pending_shot is not None else {
                "start_frame": int(event["frame"]),
                "end_frame": None,
                "outcome": None,
                "shot_type": event["type"].lower(),
                "court_x": np.nan,
                "court_y": np.nan,
            }
            row["end_frame"] = int(event["frame"])
            row["outcome"] = kind.lower()
            shots.append(row)
            pending_shot = None

if pending_shot is not None:
    pending_shot["end_frame"] = int(frame_index)
    pending_shot["outcome"] = "missed"
    shots.append(pending_shot)

shots_df = pd.DataFrame(
    shots,
    columns=["start_frame", "end_frame", "outcome", "shot_type", "court_x", "court_y"],
)
shots_df


In [ ]:
# 0	799	822	missed	jump	29.022095	27.975872
# 1	822	872	missed	jump	16.794897	45.691257
# 2	872	923	missed	jump	20.356512	4.788127
# 3	1026	1077	missed	jump	10.944176	1.191971
# 4	1231	1282	missed	layup	5.429680	23.343287
# 5	1606	1657	missed	jump	27.455963	33.086857
# 6	1835	1886	missed	jump	89.207680	8.354998

In [ ]:
made_xy = shots_df.loc[shots_df["outcome"] == "made", ["court_x", "court_y"]].to_numpy(dtype=float)
miss_xy = shots_df.loc[shots_df["outcome"] == "missed", ["court_x", "court_y"]].to_numpy(dtype=float)

made_xy = made_xy[np.isfinite(made_xy).all(axis=1)] if len(made_xy) else made_xy
miss_xy = miss_xy[np.isfinite(miss_xy).all(axis=1)] if len(miss_xy) else miss_xy

court = draw_made_and_miss_on_court(
    config=config,
    made_xy=made_xy if len(made_xy) else None,
    miss_xy=miss_xy if len(miss_xy) else None,
    miss_color=sv.Color.from_hex("#850101"),
    made_color=sv.Color.from_hex("#007A33"),
    miss_size=10,
    made_size=25,
    made_thickness=6,
    miss_thickness=6,
    line_thickness=4,
)

print(f"mapped made={len(made_xy)}  mapped miss={len(miss_xy)}")
sv.plot_image(court)


In [ ]:
made_xy = np.array([[0.0, 0.0],[0.0,50.0],[47,0],[94,50],[47,25]])
miss_xy = np.array([])

court = draw_made_and_miss_on_court(
    config=config,
    made_xy=made_xy if len(made_xy) else None,
    miss_xy=miss_xy if len(miss_xy) else None,
    miss_color=sv.Color.from_hex("#850101"),
    made_color=sv.Color.from_hex("#007A33"),
    miss_size=10,
    made_size=25,
    made_thickness=6,
    miss_thickness=6,
    line_thickness=4,
)

print(f"mapped made={len(made_xy)}  mapped miss={len(miss_xy)}")
sv.plot_image(court)

### OG Code shot detector + mapping

In [ ]:
from src.pipeline.shots import (
    filter_detections_by_team,
    predict_detection_teams,
    update_offensive_team_id,
)

BALL_IN_BASKET_CLASS_ID = 1
JUMP_SHOT_CLASS_ID = 5
LAYUP_DUNK_CLASS_ID = 6
PLAYER_IN_POSSESSION_CLASS_ID = 4
PLAYER_CLASS_ID = [0,1, 3, 4, 5, 6, 7, 9] # ball, ball_in_rim, player, player-in-possession, player-jump-shot, player-layup-dunk, player-shot-block, ball

box_annotator = sv.BoxAnnotator(color=COLOR, thickness=2)
label_annotator = sv.LabelAnnotator(color=COLOR, text_color=sv.Color.BLACK)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride = 3)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)

shot_event_tracker = ShotEventTracker(
    reset_time_frames=int(video_info.fps * 1.7),
    minimum_frames_between_starts=int(video_info.fps * 0.5),
    cooldown_frames_after_made=int(video_info.fps * 0.5),
)

offensive_team_id = None

for frame_index, frame in enumerate(frame_generator):

    # we use a RF-DETR model to detect jump shot, layup, dunk and ball in basket

    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE, iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result)
    detections = detections[np.isin(detections.class_id, PLAYER_CLASS_ID)]

    possession_dets = detections[detections.class_id == PLAYER_IN_POSSESSION_CLASS_ID]
    if len(possession_dets) > 0:
        offensive_team_id = update_offensive_team_id(
            offensive_team_id,
            predict_detection_teams(frame, possession_dets, team_classifier),
        )

    layup_dets = detections[detections.class_id == LAYUP_DUNK_CLASS_ID]
    if len(layup_dets) > 0:
        layup_teams = predict_detection_teams(frame, layup_dets, team_classifier)
        layup_dets = filter_detections_by_team(
            layup_dets, layup_teams, offensive_team_id
        )

    has_jump_shot = len(detections[detections.class_id == JUMP_SHOT_CLASS_ID]) > 0
    has_layup_dunk = len(layup_dets) > 0
    has_ball_in_basket = len(detections[detections.class_id == BALL_IN_BASKET_CLASS_ID]) > 0

    events = shot_event_tracker.update(
        frame_index=frame_index,
        has_jump_shot=has_jump_shot,
        has_layup_dunk=has_layup_dunk,
        has_ball_in_basket=has_ball_in_basket,
        layup_is_offense=has_layup_dunk,
    )

    if events:
        print(events)

        annotated_frame = frame.copy()
        annotated_frame = box_annotator.annotate(
            scene=annotated_frame,
            detections=detections)
        annotated_frame = label_annotator.annotate(
            scene=annotated_frame,
            detections=detections)



In [ ]:
config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=730, iterative_seek=True)
frame = next(frame_generator)

# we use a RF-DETR model to detect players

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE, iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
detections = sv.Detections.from_inference(result)
detections = detections[detections.class_id == PLAYER_JUMP_SHOT_CLASS_ID]

# we use a keypoint model to detect court landmarks

result = KEYPOINT_DETECTION_MODEL.infer(frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
key_points = sv.KeyPoints.from_inference(result)
landmarks_mask = key_points.keypoint_confidence[0] > KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE

if np.count_nonzero(landmarks_mask) >= 4:

    # we calculate homography matrix

    court_landmarks = np.array(config.vertices)[landmarks_mask]
    frame_landmarks = key_points[:, landmarks_mask].xy[0]

    frame_to_court_transformer = ViewTransformer(
        source=frame_landmarks,
        target=court_landmarks,
    )

    frame_xy = detections.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)

    # transform video frame coordinates into court coordinates

    court_xy = frame_to_court_transformer.transform_points(points=frame_xy)

    court = draw_made_and_miss_on_court(
        config=config,
        made_xy=court_xy,
        made_size=25,
        made_color=sv.Color.from_hex("#007A33"),
        made_thickness=6,
        line_thickness=4
    )

    sv.plot_image(court)